# Blob to Bronze: Transactions Ingestion

Continuously ingests CSV transaction files from the landing volume (`/Volumes/retail/landing_volume/transactions`) into `retail.bronze.transactions` using Auto Loader, then blocks with `awaitTermination()` so new files are picked up as they arrive.

In [0]:
# Auto Loader: read new CSV transaction files as they land in the volume
source_path = "/Volumes/retail/landing_volume/transactions"
target_table = "retail.bronze.transactions"
checkpoint_path = f"{source_path}/_checkpoints/transactions_bronze"

transactions_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("header", True)
    .load(source_path)
)

In [0]:
# Write to the bronze table.
# Serverless compute does not support continuous (ProcessingTime) streams,
# so use trigger(availableNow=True): each run processes all new files
# landed in the volume, then completes.
query = (
    transactions_df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

# Block this cell until the run finishes; re-run this cell any time to pick
# up new files and append them to retail.bronze.transactions.
query.awaitTermination()

In [0]:
%sql
SELECT * FROM retail.bronze.transactions